# 🏎️ F1 Season Monte Carlo Simulator — Results Success Testing

This notebook loads the output CSVs from the Monte Carlo simulation and tests the predicted results against the actual results.

Each CSV represents 10,000+ simulated seasons. Values in each cell are the **probability** (0–1) that a given driver finishes the championship in that position.

---

In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install fastf1
%pip install scikit-learn

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import fastf1
import os
from sklearn.calibration import calibration_curve
from datetime import datetime

year=datetime.now().year
current_directory = os.getcwd()

#Create project Directory Path
PROJECT_DIR = Path(current_directory).resolve().parent

#create Cache
CACHE_DIR=PROJECT_DIR / "fastf1_cache"
fastf1.Cache.enable_cache(CACHE_DIR)

#OUTPUT_DIR = Path("..") / f"csv_files/{next_race}"
GRAPH_DIR = Path("..") / f"graph_files/race_analysis"
CACHE_DIR = Path("..") / f"fastf1_cache/{year}"

CSV_DIR = Path("..") / f"csv_files"


# Plot styling
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#2a2a2a'
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.dpi'] = 120

F1_RED = '#E8002D'
F1_SILVER = '#C0C0C0'

print('Setup complete.')

## 1. Load Data

In [ ]:
def load_sim_csv(filename):
    """Load a simulation output CSV. Index = driver codes, columns = finishing positions + DNF."""
    df = pd.read_csv( filename, index_col=0)
    # Ensure column names are strings for consistent handling
    df.columns = df.columns.astype(str)
    return df

predicted_happened_races=[]
cached_races=[]

next_prediction_post=None

current_year=datetime.now().year
schedule = fastf1.get_event_schedule(current_year)

for gp_dir in CACHE_DIR.iterdir():

    race_folders = list(gp_dir.glob("*_Race"))

    if not race_folders:
        continue

    race_folder = race_folders[0]

    if any(race_folder.iterdir()):
        print(f"Race cached: {gp_dir.name}")
        cached_races.append(gp_dir)
    else:
        continue
        #print(f"No race data: {gp_dir.name}")

print(cached_races)
for race_file in CSV_DIR.rglob("*_race.csv"):
    df = load_sim_csv(race_file)
    gp_name = race_file.parent.name
    formatted_gp_name=gp_name.replace(" ", "_")

    if any(formatted_gp_name in str(path) for path in cached_races):
        print("Race exists in cache")
        index_id=schedule.loc[schedule["EventName"]==gp_name,["RoundNumber"]]
        pre_true=race_file.name.startswith('pre')
        if(pre_true):
            index_id=str(index_id)[-1]+"_1pre"
        else:
            index_id=str(index_id)[-1]+"_2post"
        predicted_happened_races.append((race_file,gp_name,index_id))
    else:
        print(f"{gp_name} not in {cached_races}")
        if(race_file.name.startswith('pre')):
            next_prediction_pre=race_file
            next_prediction_pre=load_sim_csv(next_prediction_pre)
            print(next_prediction_pre)
        else:
            next_prediction_post=race_file
            next_prediction_post=load_sim_csv(next_prediction_post)
            print(next_prediction_post)

sorted_races=sorted(predicted_happened_races, key=lambda x: x[2])

## 2. Double Check sorted_races

In [ ]:
#ADD STUFF HERE LATER

race_results={}

for path,name,index_id in sorted_races:
    session=fastf1.get_session(year,name,"R")
    session.load(
        laps=False,
        telemetry=False,
        weather=False,
        messages=False
        )
    result=session.results
    if(result.empty):
        print(f"{name} RACE HASN'T HAPPENED")
        sorted_races.remove((path,name,index_id))
    else:
        race_results[name] = {
            row["Abbreviation"]: row["ClassifiedPosition"]
            for _, row in result.iterrows()
        }

print(race_results)


## 3. RACE RESULT PREDICTION METRICS

Log Loss, Brier, and expected vs actual

In [ ]:
def race_log_loss(df, actual):

    # align drivers
    df = df.loc[actual.index]

    cols = df.columns
    col_idx = cols.get_indexer(actual.values)
    row_idx = np.arange(len(df))

    probs = df.to_numpy()[row_idx, col_idx]

    log_loss = -np.log(probs + 1e-15)

    return pd.DataFrame({
        "Driver": df.index,
        "Actual": actual.values,
        "Prob": probs,
        "LogLoss": log_loss
    })

def per_driver_brier(df, actual):

    cols = df.columns
    col_idx = cols.get_indexer(actual.values)
    row_idx = np.arange(len(df))

    probs = df.to_numpy()[row_idx, col_idx]

    brier = (1 - probs) ** 2

    return pd.DataFrame({
        "Driver": df.index,
        "Actual": actual.values,
        "Prob": probs,
        "Brier": brier
    })

def expected_vs_actual(df, actual):

    finish_mask = actual != "DNF"

    df_finish = df.loc[finish_mask]
    actual_finish = actual[finish_mask]

    finish_cols = [c for c in df.columns if c != "DNF"]

    positions = np.array([int(c) for c in finish_cols])

    expected_finish = df[finish_cols].to_numpy() @ positions
    actual_finish = pd.to_numeric(
        actual.replace("DNF", np.nan),
        errors="coerce"
    ).values

    abs_error = np.where(
        ~np.isnan(actual_finish),
        np.abs(expected_finish - actual_finish),
        np.nan
    )

    return pd.DataFrame({
        "Driver": df.index,
        "ExpectedFinish": expected_finish,
        "ActualFinish": actual_finish,
        "AbsError": abs_error
    })

def expected(df):
    finish_cols = [c for c in df.columns if c != "DNF"]

    positions = np.array([int(c) for c in finish_cols])

    expected_finish = df[finish_cols].to_numpy() @ positions

    return pd.DataFrame({
        "Driver": df.index,
        "ExpectedFinish": expected_finish
    })


log_losses={}
briers={}
evas={}

for path,name,index_id in sorted_races:
    print(name)
    predictive_df=pd.read_csv(path)

    df = predictive_df.rename(columns={"Unnamed: 0": "Driver"}).copy()
    df.columns = df.columns.astype(str)
    df = df.set_index("Driver")

    actual_dict=race_results[name]

    actual = pd.Series(actual_dict).astype(str).replace("R", "DNF")

    df = df.loc[actual.index]

    file_stem=Path(path).stem

    full_index=name+"_"+file_stem

    log_losses[full_index]=(race_log_loss(df, actual))
    briers[full_index]=(per_driver_brier(df, actual))
    evas[full_index]=(expected_vs_actual(df, actual))

## 4. FULL SEASON RACE BY RACE LOG LOSS

Visualize the log loss trend over the season

In [ ]:
def log_loss_graph(df,name):
    log_df = pd.Series(df).reset_index()
    log_df.columns = ["Race", "LogLoss"]

    log_df["Rolling"] = log_df["LogLoss"].rolling(3).mean()

    plt.figure(figsize=(10,5))
    plt.plot(log_df["Race"], log_df["LogLoss"], alpha=0.5,color=F1_SILVER)
    plt.plot(log_df["Race"], log_df["Rolling"], linewidth=3,color=F1_RED)
    plt.plot(log_df["Race"], log_df["LogLoss"], marker="o",color=F1_SILVER)
    plt.xticks(rotation=45)
    plt.ylabel("Log Loss")
    plt.xlabel("Race")
    plt.title(f"{name} Race-by-Race Log Loss")
    plt.tight_layout()

    FOLDER_PATH=GRAPH_DIR /'race_v_seasons'

    FOLDER_PATH.mkdir(parents=True, exist_ok=True)
    plt.savefig(FOLDER_PATH / f"{name} Race-by-Race Log Loss.png", dpi=150,
        bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()

def driver_metric(metric_dict, driver, metric):
    return pd.Series({
        race: df.loc[df["Driver"] == driver, metric].iloc[0]
        for race, df in metric_dict.items()
    })

def driver_metrics(metric_dict, driver, metrics=None):
    if metrics is None:
        metrics = []

    return pd.Series({
        race: df.loc[df["Driver"] == driver, metrics].iloc[0]
        for race, df in metric_dict.items()
    })

all_logloss = pd.concat(log_losses.values(), ignore_index=True)
all_briers = pd.concat(briers.values(), ignore_index=True)

race_logloss = pd.Series({
    race: df["LogLoss"].mean()
    for race, df in log_losses.items()
})

log_loss_graph(race_logloss,"Combined")

for driver in df.index:
    driver_log=driver_metric(log_losses,driver,"LogLoss")
    log_loss_graph(driver_log,driver)

## 5. Prediction Error by Driver

Visualize each drivers Absolute Error

In [ ]:
all_evas = pd.concat(evas.values(), ignore_index=True)

driver_errors = (
    all_evas.groupby("Driver")["AbsError"]
    .mean()
    .sort_values()
)

max_idx = np.argmax(driver_errors)
colors = [F1_RED if i == max_idx else F1_SILVER for i in range(len(driver_errors))]
plt.figure(figsize=(8,6))
driver_errors.plot(kind="barh",color=colors)

plt.xlabel("Average Absolute Error")
plt.ylabel("Driver")
plt.title("Prediction Error by Driver")
plt.tight_layout()
FOLDER_PATH=GRAPH_DIR /'Predicted_Error'

FOLDER_PATH.mkdir(parents=True, exist_ok=True)
plt.savefig(FOLDER_PATH / f"{sorted_races[-1][-1]} Predicted_Error_by_Driver.png", dpi=150,
    bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

## 6. Full Season Expected vs Actual Graph

Visualize the relationship between the expected finish and the actual finish

In [ ]:
def expect_vs_actual(eva_list,name,next_prediction_post=None,next_prediction_pre=None):
    plt.figure(figsize=(8,8))
    plt.scatter(
        eva_list["ExpectedFinish"],
        eva_list["ActualFinish"],
        alpha=0.6,
        color=F1_SILVER
    )

    max_pos = max(
        eva_list["ExpectedFinish"].max(),
        eva_list["ActualFinish"].max()
    )

    plt.plot(
        [1, max_pos],
        [1, max_pos],
        linestyle="--",
        color=F1_RED
    )
    if(name!="FULL"):
        expected_value=(next_prediction_pre[next_prediction_pre["Driver"]==driver]["ExpectedFinish"].values[0])
        plt.plot(
            [expected_value,expected_value],
            [1,max_pos],
            linestyle="--"
        )
    
        if(next_prediction_post is not None):
            expected_value_post=(next_prediction_pre[next_prediction_pre["Driver"]==driver]["ExpectedFinish"].values[0])
            plt.plot(
                [expected_value_post,expected_value_post],
                [1,max_pos],
                linestyle="--"
            )

    plt.xlabel("Expected Finish")
    plt.ylabel("Actual Finish")
    plt.title(f"{name} Expected vs Actual Finish Position")
    FOLDER_PATH=GRAPH_DIR /'Expected_vs_Actual'/f"{sorted_races[-1][-1]}"

    FOLDER_PATH.mkdir(parents=True, exist_ok=True)
    plt.savefig(FOLDER_PATH / f"{sorted_races[-1][-1]}_{name} Expected_vs_Actual.png", dpi=150,
        bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()

expect_vs_actual(all_evas,"FULL")

for driver in df.index:
    driver_log=driver_metrics(evas,driver,["ExpectedFinish","ActualFinish"])
    driver_log = pd.DataFrame(driver_log.tolist(), index=driver_log.index)
    if(next_prediction_post is None):
        exepected_results_pre=expected(next_prediction_pre)
        expect_vs_actual(driver_log,driver,next_prediction_pre=exepected_results_pre)
    else:
        exepected_results_pre=expected(next_prediction_pre)
        exepected_results_post=expected(next_prediction_post)
        expect_vs_actual(driver_log,driver,next_prediction_pre=exepected_results_pre,next_prediction_post=exepected_results_post)

## 7. Expected vs Actual Finish Difference Graph

Visualize the trends in error by expected finish position

In [ ]:
all_evas["Residual"] = (
    all_evas["ActualFinish"]
    - all_evas["ExpectedFinish"]
)

plt.scatter(
    all_evas["ExpectedFinish"],
    all_evas["Residual"]
)

plt.axhline(0, linestyle="--",color=F1_RED)
plt.xlabel("Expected Finish")
plt.ylabel("Actual Finish")
plt.title("Expected vs Actual Finish Position")
FOLDER_PATH=GRAPH_DIR /'Expected_Diff'

FOLDER_PATH.mkdir(parents=True, exist_ok=True)
plt.savefig(FOLDER_PATH / f"{sorted_races[-1][-1]} Expected_Diff.png", dpi=150,
    bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

plt.scatter(
    all_evas["ActualFinish"],
    all_evas["Residual"]
)

plt.axhline(0, linestyle="--",color=F1_RED)
plt.xlabel("Actual Finish")
plt.ylabel("Expected Finish")
plt.title("Actual vs Expected Finish Position")
FOLDER_PATH=GRAPH_DIR /'Actual_Diff'

FOLDER_PATH.mkdir(parents=True, exist_ok=True)
plt.savefig(FOLDER_PATH / f"{sorted_races[-1][-1]} Actual_Diff.png", dpi=150,
    bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

## 8. Previous Race Probability

In [ ]:
odds={}
y_axis1={}
y_axis2={}
for path,name,index_id in sorted_races:
    odds[name]={}
    y_axis1[name]={}
    y_axis2[name]={}
    odds[name]["guessed"]={}
    odds[name]["missed"]={}
    predictive_df=pd.read_csv(path)

    df = predictive_df.rename(columns={"Unnamed: 0": "Driver"}).copy()
    df.columns = df.columns.astype(str)
    df = df.set_index("Driver")

    actual_dict=race_results[name]

    actual = pd.Series(actual_dict).astype(str).replace("R", "DNF").astype(str).replace("W", "DNF")

    df = df.loc[actual.index]
    race_odds=1
    race_miss=0
    drive_count=0

    for driver in df.index:
        real=(actual[driver])
        prob_=df.loc[driver][real]
        race_odds+=prob_
        drive_count+=1
        if(prob_==0):
            race_miss+=1
        
    race_odds=race_odds/drive_count

    odds[name]["guessed"]=race_odds
    odds[name]["missed"]=race_miss
    y_axis1[name]=race_odds
    y_axis2[name]=race_miss

y1 = list(y_axis1.values())
y2 = list(y_axis2.values())

x_axis = list(y_axis1.keys())
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Race Prediction Success')
ax1.set_ylabel('Average Final Pos Prob',color=color)
ax1.plot(x_axis,y1,color=color, alpha=0.5)
ax1.tick_params(axis='y', labelcolor=color)

ax2=ax1.twinx()

color = F1_SILVER
ax2.set_ylabel('Race Prediction Misses',color=color)
ax2.plot(x_axis,y2,color=color, alpha=0.5)
ax2.tick_params(axis='y', labelcolor=color)

ax1.tick_params(axis='x',rotation=45)
plt.title("Race Result Probability")
FOLDER_PATH=GRAPH_DIR /'Race_Probability'

FOLDER_PATH.mkdir(parents=True, exist_ok=True)
plt.savefig(FOLDER_PATH / f"Finish Probability.png", dpi=150,
    bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---

## Summary

| Chart | File | Description |
|---|---|---|
| Race vs Season Scatter | `race_vs_season.png` | Race win % vs championship win % per driver |

All charts are saved to `output/` and can be used directly in LinkedIn posts or reports.